# 1. Data cleaning

**Data:** NHIS 2024 Sample Adult file (CDC), one row per respondent.
**Output:** `data/processed/clean_dataset.csv`, one row per student aged 18-29, used by every later notebook.

Sections: 1.1 setup, 1.2 the sample, 1.3 work (Jørgen), 1.4 sleep, sex and activity (Kristine),
1.5 mental health, children and education (Endre), 1.6 economy and health (Lykke), 1.7 checks and save.

NHIS marks "refused / not ascertained / don't know" with special codes (7/8/9, or 97/98/99).
These are turned into missing values (NaN). The variables and codes are listed in `docs/variables.md`.

## 1.1 Setup

In [1]:
import sys

import numpy as np
import pandas as pd
from IPython.display import Image

from src.config import PROCESSED, ROOT
from src.data import load_nhis

print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)

CODEBOOK = ROOT / "docs" / "codebook"  # screenshots of the NHIS codebook
OUT_FILE = PROCESSED / "clean_dataset.csv"

# Every NHIS column used in this notebook, by section
COLUMNS = {
    "sample": ["HHX", "AGEP_A", "SCHCURENR_A"],
    "work": ["WTFA_A", "EMPWRKLSW1_A", "EMPWKHRS3_A"],
    "sleep": ["SLPHOURS_A", "SLPREST_A", "SLPFLL_A", "SLPSTY_A", "SEX_A", "PA18_02R_A"],
    "mental_health": ["ANXFREQ_A", "DEPFREQ_A", "K6SPD_A", "PCNTKIDS_A", "EDUCP_A"],
    "economy_health": ["RATCAT_A", "FDSCAT4_A", "HOUSECOST_A", "HYPEV_A", "ASEV_A", "ARTHEV_A", "DIBEV_A"],
}
raw = load_nhis(usecols=[c for cols in COLUMNS.values() for c in cols])  # downloads the file if missing
print(raw.shape)


def to_missing(series, codes):
    """Replace NHIS special codes (refused / not ascertained / don't know) with NaN."""
    return series.mask(series.isin(codes))

Python 3.14.7 | pandas 3.1.1 | numpy 2.5.4


(32629, 24)


## 1.2 The sample: students aged 18-29

A row is removed only when it fails a sample criterion. Students with unknown school status (7/8/9)
cannot be placed, so they are left out.

In [2]:
by_age = raw[raw["AGEP_A"].between(18, 29)]  # also drops the age codes 97/99
unknown_school = by_age["SCHCURENR_A"].isin([7, 8, 9]).sum()
students = by_age[by_age["SCHCURENR_A"] == 1].reset_index(drop=True)

print("All respondents:          ", len(raw))
print("Aged 18-29:               ", len(by_age))
print("Unknown school status:    ", unknown_school, "(left out)")
print("Students aged 18-29:      ", len(students))

# Every section below adds its columns to this table, so all columns share the same rows
clean = pd.DataFrame({"household_id": students["HHX"], "age": students["AGEP_A"]})

## 1.3 Work hours (Jørgen)

- `survey_weight` is kept for population estimates.
- "Don't know" codes 97/98/99 become missing, and the row is kept.
- An empty value means the question was not asked: students who did not work last week (code 2) get 0 hours.
- Unknown work status (code 9) does not get 0 hours; it stays missing.

In [3]:
hours = students["EMPWKHRS3_A"]
print("Hours set to missing (97/98/99):", hours.isin([97, 98, 99]).sum())
hours = to_missing(hours, [97, 98, 99])

did_not_work = students["EMPWRKLSW1_A"] == 2
print("Did not work, set to 0 hours:   ", did_not_work.sum())
hours = hours.mask(did_not_work, 0)

clean["survey_weight"] = students["WTFA_A"]
clean["weekly_work_hours"] = hours.astype("Int64")
clean[["survey_weight", "weekly_work_hours"]].describe().round(1)

## 1.4 Sleep, sex and physical activity (Kristine)

- `sleep_hours_per_night`: hours of sleep per 24 hours; 97/98/99 become missing.
- The three sleep items (last 30 days) are labelled Never / Some Days / Most days / Every Day.
- Sex: 7/9 become missing. Physical activity: 8 becomes missing.

In [4]:
FREQUENCY = {1: "Never", 2: "Some Days", 3: "Most days", 4: "Every Day"}

clean["sleep_hours_per_night"] = to_missing(students["SLPHOURS_A"], [97, 98, 99])
clean["past_moth_felt_rested"] = to_missing(students["SLPREST_A"], [7, 8, 9]).map(FREQUENCY)
clean["past_moth_trouble_falling_asleep"] = to_missing(students["SLPFLL_A"], [7, 8, 9]).map(FREQUENCY)
clean["past_moth_trouble_staying_asleep"] = to_missing(students["SLPSTY_A"], [7, 8, 9]).map(FREQUENCY)
clean["sex"] = to_missing(students["SEX_A"], [7, 9]).map({1: "Male", 2: "Female"})
clean["activity_guideline_status"] = to_missing(students["PA18_02R_A"], [8]).map(
    {1: "Inactive", 2: "Insufficiently active", 3: "Sufficiently active"}
)

print(clean["sleep_hours_per_night"].value_counts().sort_index().to_string())

In [5]:
for column in ["past_moth_felt_rested", "past_moth_trouble_falling_asleep", "past_moth_trouble_staying_asleep",
               "sex", "activity_guideline_status"]:
    print(clean[column].value_counts(dropna=False).to_string(), "\n")

## 1.5 Mental health, children and education (Endre)

The codes are kept as numbers; the codebook screenshots show what each number means.

In [6]:
for nhis, name in {
    "ANXFREQ_A": "anxiety_frequency",
    "DEPFREQ_A": "depression_frequency",
    "K6SPD_A": "serious_distress",
    "PCNTKIDS_A": "kids_household",
}.items():
    clean[name] = to_missing(students[nhis], [7, 8, 9])
clean["education_level"] = to_missing(students["EDUCP_A"], [97, 98, 99])

clean[["anxiety_frequency", "depression_frequency", "serious_distress", "kids_household",
       "education_level"]].apply(lambda s: s.value_counts(dropna=False)).sort_index()

In [7]:
display(Image(filename=CODEBOOK / "ANXFREQ_A_explained.png"))
display(Image(filename=CODEBOOK / "DEPFREQ_A_explained.png"))
display(Image(filename=CODEBOOK / "K6SPD_A_explained.png"))

In [8]:
display(Image(filename=CODEBOOK / "PCNTKIDS_A_explained.png"))
display(Image(filename=CODEBOOK / "EDUCP_A_explained.png"))

## 1.6 Economy and health (Lykke)

- `family_income` (income-to-poverty category) has no special codes in the sample.
- 7/8/9 become missing for the other columns. The yes/no items are labelled YES / NO.
- `food_security_level` is a readable copy of `food_security`.

In [9]:
clean["family_income"] = students["RATCAT_A"]
clean["food_security"] = to_missing(students["FDSCAT4_A"], [7, 8, 9])
for nhis, name in {
    "HOUSECOST_A": "housing_cost",
    "HYPEV_A": "high_bloodpressure",
    "ASEV_A": "had_asthma",
    "ARTHEV_A": "had_arthritis",
    "DIBEV_A": "had_diabetes",
}.items():
    clean[name] = to_missing(students[nhis], [7, 8, 9]).map({1: "YES", 2: "NO"})
clean["food_security_level"] = clean["food_security"].map(
    {1: "High", 2: "Marginal", 3: "Low", 4: "Very Low"}
)

for column in ["food_security_level", "housing_cost", "high_bloodpressure", "had_asthma",
               "had_arthritis", "had_diabetes"]:
    print(clean[column].value_counts(dropna=False).to_string(), "\n")

## 1.7 Checks and save

In [10]:
assert len(clean) == 1085, "the sample size has changed"
assert clean["household_id"].is_unique, "household_id must identify one student"
assert clean["age"].between(18, 29).all()
assert clean["weekly_work_hours"].notna().all() and clean["weekly_work_hours"].max() < 97
assert clean["sleep_hours_per_night"].notna().all() and clean["sleep_hours_per_night"].max() < 97

missing = clean.isna().sum()
print(clean.shape)
print("Missing values per column:")
print(missing[missing > 0].to_string())

In [11]:
PROCESSED.mkdir(parents=True, exist_ok=True)
clean.to_csv(OUT_FILE, index=False)
print("Saved", clean.shape, "to", OUT_FILE.relative_to(ROOT))
clean.head()